In [4]:
def time_to_seconds(t):
    if pd.isna(t):
        return 0.0
    h, m, s = t.split(":")
    return int(h) * 3600 + int(m) * 60 + float(s)

In [5]:
import os
import pandas as pd
import subprocess

# === ПУТИ ===
CSV_PATH = "train_soundscapes_labels.csv"
AUDIO_DIR = "train_soundscapes"
OUTPUT_DIR = "chunks"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# === 1. Читаем CSV ===
df = pd.read_csv(CSV_PATH, dtype={"primary_label": str})

# === 2. Разбиваем источники ===
df["labels_list"] = df["primary_label"].apply(
    lambda x: x.split(";") if pd.notna(x) else []
)

# === 3. функция ffmpeg ===
def cut_audio(input_file, output_file, start, duration):
    cmd = [
        "ffmpeg",
        "-y",
        "-i", input_file,
        "-ss", str(start),
        "-t", str(duration),
        "-c", "copy",
        output_file
    ]
    subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# === 4. Основной цикл ===
files = []
sources_per_file = []

for _, row in df.iterrows():
    filename = row["filename"]
    start = time_to_seconds(row["start"])
    end = time_to_seconds(row["end"])
    labels = row["labels_list"]

    input_path = os.path.join(AUDIO_DIR, filename)
    if not os.path.exists(input_path):
        continue

    duration = end - start

    # имя нового файла
    base = filename.replace(".ogg", "")
    chunk_name = f"{base}_{int(start)}_{int(end)}.ogg"
    output_path = os.path.join(OUTPUT_DIR, chunk_name)

    # === режем ===
    cut_audio(input_path, output_path, start, duration)

    # === сохраняем в списки ===
    files.append(chunk_name)
    sources_per_file.append(labels)

# # === 5. Результат ===
# print("Количество кусков:", len(files))
# print("\nПример:")
#
# for i in range(5):
#     print(files[i], "->", sources_per_file[i])

# === 6. Итоговая структура ===
result = {
    "files": files,
    "sources_per_file": sources_per_file
}

print("\nГотово ✅")

Количество кусков: 1478

Пример:
BC2026_Train_0039_S22_20211231_201500_0_5.ogg -> ['22961', '23158', '24321', '517063', '65380']
BC2026_Train_0039_S22_20211231_201500_5_10.ogg -> ['22961', '23158', '24321', '517063', '65380']
BC2026_Train_0039_S22_20211231_201500_10_15.ogg -> ['22961', '23158', '24321', '517063', '65380']
BC2026_Train_0039_S22_20211231_201500_15_20.ogg -> ['22961', '23158', '24321', '517063', '65380']
BC2026_Train_0039_S22_20211231_201500_20_25.ogg -> ['22961', '23158', '24321', '517063', '65380']

Готово ✅


✅ список .ogg файлов
✅ для каждого файла — список источников (ID: буквенно-цифровые коды)
❗ неизвестны громкости и сдвиги

In [11]:
import pandas as pd
from collections import Counter

# === 1. ВАЖНО: читаем primary_label как строку ===
df = pd.read_csv(
    "train_soundscapes_labels.csv",
    dtype={"primary_label": str}
)

# === 2. Разбиваем список ID ===
df["primary_label_list"] = df["primary_label"].apply(
    lambda x: x.split(";") if pd.notna(x) else []
)

# === 3. Уникальные файлы ===
filenames = df["filename"].unique().tolist()
print("Количество файлов:", len(filenames))

# === 4. Собираем источники по файлам ===
file_to_sources = {}

for _, row in df.iterrows():
    fname = row["filename"]
    labels = row["primary_label_list"]

    if fname not in file_to_sources:
        file_to_sources[fname] = set()

    file_to_sources[fname].update(labels)

# === 5. Приводим к спискам ===
file_to_sources = {
    k: sorted(list(v)) for k, v in file_to_sources.items()
}

# === 6. Все источники ===
all_sources = []
for labels in df["primary_label_list"]:
    all_sources.extend(labels)

unique_sources = sorted(set(all_sources))
source_counts = Counter(all_sources)

# === 7. Проверка ===
print("\nПример:")
for i, (k, v) in enumerate(file_to_sources.items()):
    print(k, "->", v)
    if i >= 4:
        break

print("\nУникальных источников:", len(unique_sources))

print("\nТоп-10:")
for label, count in source_counts.most_common(10):
    print(label, count)

Количество файлов: 66

Пример:
BC2026_Train_0039_S22_20211231_201500.ogg -> ['22961', '23158', '24321', '517063', '65380']
BC2026_Train_0019_S22_20211104_200000.ogg -> ['22973', '47144', '517063', '65380', 'bunibi1']
BC2026_Train_0053_S22_20220211_204500.ogg -> ['22973', '24321', '517063', '555146', '65380', '66971']
BC2026_Train_0047_S22_20220131_003000.ogg -> ['22973', '23158', '24279', '517063', '555146', '65380']
BC2026_Train_0027_S22_20211129_014500.ogg -> ['1491113', '22967', '22973', '24279', '517063', '65380', '67252', 'trsowl']

Уникальных источников: 75

Топ-10:
65380 666
517063 626
22973 426
555146 420
23158 350
24279 346
24321 344
22967 310
66971 298
47158son25 168


НАПРИМЕР

In [ ]:
files = [
    "file1.ogg",
    "file2.ogg",
    "file3.ogg",
]

sources_in_files = [
    ["A12", "B7"],     # file1
    ["B7", "C99"],     # file2
    ["C99", "A12"],    # file3
]

Преобразование ID → индексы

Алгоритму нужны числа, не строки.

In [ ]:
def build_source_index(sources_in_files):
    unique_sources = sorted(set(s for lst in sources_in_files for s in lst))
    source_to_idx = {s: i for i, s in enumerate(unique_sources)}
    return source_to_idx, unique_sources

source_to_idx = {
    "A12": 0,
    "B7": 1,
    "C99": 2
}

👉 каждому источнику присваивается индекс

Построение A_mask

In [ ]:
import numpy as np

def build_A_mask(sources_in_files, source_to_idx):
    M = len(sources_in_files)
    N = len(source_to_idx)

    A_mask = np.zeros((M, N), dtype=np.float32)

    for i, src_list in enumerate(sources_in_files):
        for s in src_list:
            j = source_to_idx[s]
            A_mask[i, j] = 1.0

    return A_mask

Пример результата
A_mask =
[
 [1, 1, 0],
 [0, 1, 1],
 [1, 0, 1],
]

1. Загрузка и STFT

In [ ]:
import numpy as np
import librosa
import soundfile as sf
# загрузка ogg файлов
"""librosa.load(f, sr=32000, mono=True)
читает .ogg напрямую (через встроенный декодер)
важно:
автоматически декодирует OGG → PCM
приводит к частоте 32000 Гц
переводит в моно"""
def load_signals(files, sr=32000):
    signals = [librosa.load(f, sr=sr, mono=True)[0] for f in files]
    # обрезаем все сигналы до одной длины
    # STFT требует одинаковые размеры
    # иначе матрицы не сложатся
    min_len = min(len(s) for s in signals)
    signals = [s[:min_len] for s in signals]
    # M — число файлов
    # T — длина сигнала
    # signals.shape = (M, T)
    return np.array(signals), sr

# STFT — переход в частоты
# Каждый сигнал превращается в:
# (F, T) комплексная матрица
# где:
# F — частоты
# T — временные окна
def compute_stft(signals, n_fft=1024, hop=256):
    stfts = [librosa.stft(s, n_fft=n_fft, hop_length=hop) for s in signals]
    # X.shape = (F, T, M)
    # где:
    # M — количество файлов
    X = np.stack(stfts, axis=-1)  # (F, T, M)
    return X

2. Основной алгоритм (быстрый NMF с маской)

In [ ]:
def masked_nmf(V, A_mask, n_iter=30):
    """
    V: (F, T, M) — амплитуды
    A_mask: (M, N) — 0/1 (структура источников)
    """
    F, T, M = V.shape
    N = A_mask.shape[1]

    # --- инициализация ---
    # W	спектры источников
    # H	активность во времени
    # G	громкости источников в файлах
    W = np.abs(np.random.randn(F, N)) + 1e-6
    H = np.abs(np.random.randn(N, T)) + 1e-6
    G = (np.abs(np.random.randn(M, N)) + 1e-6) * A_mask

    for _ in range(n_iter):

        # --- реконструкция (векторизировано) ---
        # (F, N) x (N, T) → (F, T)
        WH = np.einsum('fn,nt->fnt', W, H)  # (F, N, T)

        # учитываем G и A_mask
        # каждый источник добавляется только туда, где разрешён
        # с коэффициентом G
        V_hat = np.einsum('mnt,fnt->ftm', G, WH)  # (F, T, M)
        V_hat = np.maximum(V_hat, 1e-8)

        # --- обновление H ---
        num = np.einsum('ftm,mn,fn->nt', V / V_hat, G, W)
        den = np.einsum('mn,fn->nt', G, W) + 1e-8
        H *= num / den

        # --- обновление W ---
        num = np.einsum('ftm,mn,nt->fn', V / V_hat, G, H)
        den = np.einsum('mn,nt->fn', G, H) + 1e-8
        W *= num / den

        # --- обновление G ---
        num = np.einsum('ftm,fnt->mn', V / V_hat, WH)
        den = np.einsum('fnt->mn', WH) + 1e-8
        # умножение на A_mask
        # запрещает появление источников там, где их нет
        G *= (num / den) * A_mask  # соблюдаем структуру

    return W, H, G

3. Восстановление источников (ключевой момент)

In [ ]:
def reconstruct_sources(X, W, H):
    """
    X: комплексный STFT (F, T, M)
    """
    F, T, M = X.shape
    N = H.shape[0]

    # спектры источников
    S_mag = np.einsum('fn,nt->fnt', W, H)  # (F, N, T)

    # soft mask
    total = np.sum(S_mag, axis=1, keepdims=True) + 1e-8
    masks = S_mag / total  # (F, N, T)

    # используем фазу первого канала
    phase = np.exp(1j * np.angle(X[:, :, 0]))

    sources = []
    for i in range(N):
        S_complex = masks[:, i, :] * np.abs(X[:, :, 0]) * phase
        s = librosa.istft(S_complex, hop_length=256)
        sources.append(s)

    return sources

n_fft
n_fft = 2048  - лучше качество

🔹 hop_length
hop = 256 или 512

🔹 n_iter
30 → быстро
80 → качественно
4. Полный pipeline

In [ ]:
def separate(files, A_mask):
    signals, sr = load_signals(files)
    X = compute_stft(signals)

    V = np.abs(X)

    W, H, G = masked_nmf(V, A_mask, n_iter=30)

    sources = reconstruct_sources(X, W, H)

    return sources, sr

5. Сохранение

In [ ]:
# sources, sr = separate(files, A_mask)
#
# for i, s in enumerate(sources):
#     sf.write(f"source_{i}.ogg", s, sr)

In [ ]:
def separate_pipeline(files, sources_in_files):
    signals, sr = load_signals(files)

    source_to_idx, idx_to_source = build_source_index(sources_in_files)
    A_mask = build_A_mask(sources_in_files, source_to_idx)

    X = compute_stft(signals)
    V = np.abs(X)

    W, H = masked_nmf(V, A_mask, n_iter=50)

    sources = reconstruct_sources(X, W, H)

    return sources, idx_to_source, sr

In [ ]:
import soundfile as sf

sources, names, sr = separate_pipeline(files, sources_in_files)

for s, name in zip(sources, names):
    sf.write(f"{name}.ogg", s, sr)

Как улучшить ещё
🚀 1. Увеличить FFT
n_fft = 2048

→ лучше частотное разделение

🚀 2. Больше итераций
n_iter = 50–100
🚀 3. Лог-спектр
V = np.log1p(np.abs(X))
🚀 4. GPU (очень советую)

Могу переписать на PyTorch → ускорение ×10

🔴 Ограничения (честно)
если 2 источника очень похожи → сложнее
если один всегда доминирует → второй хуже восстанавливается